# seis-sgm: S0/S1 bootstrap
Runtime > Change runtime type > pick a GPU (T4 is fine for S1), then run top to bottom.

In [2]:
!nvidia-smi -L || echo "no GPU on this runtime"

GPU 0: Tesla T4 (UUID: GPU-5be62636-2282-7921-602d-9b3ff7d7ca97)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, subprocess
REPO = "MBS-V/seis-sgm"  # <-- edit this
DEST = "/content/seis-sgm"
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")       # Colab Secret; only needed for a private repo
except Exception:
    tok = None
url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, DEST], check=True)
%cd /content/seis-sgm

/content/seis-sgm


In [9]:
!pip -q install segyio h5py
!python scripts/fetch_vendor.py

[..] fetching IPS-RNA @ eed7523e (shallow)
[ok] IPS-RNA at eed7523e
[..] fetching VI-Non-IID @ 6a94b8d1 (shallow)
[ok] VI-Non-IID at 6a94b8d1


## S1 checks (paste each output back into the chat)

In [10]:
!python scripts/s1_checks.py env


===== env =====
python 3.13.16 on Linux
torch 2.14.1+cu130 | cuda available: True
GPU: Tesla T4, 14.6 GiB, compute capability 7.5
missing packages: none
vendor/IPS-RNA: pinned OK
vendor/VI-Non-IID: pinned OK
sampler from: vendor/IPS-RNA/models/__init__.py
VDN from:     vendor/IPS-RNA/utils/networks/VDN.py
weights sha256[:16]: c3beee608d08188a (OK)


In [11]:
!python scripts/s1_checks.py sigma


===== sigma =====
device=cuda  patch std=0.367
   true   median      min      max  rel.err
 0.0500   0.0542   0.0254   0.0777     8.5%
 0.1000   0.1025   0.0702   0.1465     2.5%
 0.1819   0.1849   0.1155   0.2441     1.7%


In [12]:
!python scripts/s1_checks.py gauss


===== gauss =====
index of sigma: 316 sigma_index= 0.18129922
general annealed Langevin sampling:   0% 0/184 [00:00<?, ?it/s]noise index in plot: 316 sigmas[index]: 0.18129922
general annealed Langevin sampling:   1% 1/184 [00:00<00:49,  3.69it/s]noise index in plot: 362 sigmas[index]: 0.08751406
general annealed Langevin sampling:  32% 59/184 [00:00<00:00, 202.35it/s]noise index in plot: 408 sigmas[index]: 0.042243484
general annealed Langevin sampling:  56% 103/184 [00:00<00:00, 278.87it/s]noise index in plot: 454 sigmas[index]: 0.020391146
general annealed Langevin sampling:  81% 149/184 [00:00<00:00, 335.66it/s]499
general annealed Langevin sampling: 100% 184/184 [00:00<00:00, 282.54it/s]

device=cuda  sigma_y=0.1819  start index Ln=316 (sigma_Ln=0.181299)  n=64 samples
                                       shrink   target       var    target  finite
A authors' code (sigma_y in 1/(..))    0.7652   0.7312   0.02302   0.02409  True
B paper-literal (sigma_Ln, Ln+1..)     0.7662   0.

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [5]:
import os

print("Current project directory exists:", os.path.isdir("/content/seis-sgm"))
print("\nProject contents:")

for item in sorted(os.listdir("/content/seis-sgm")):
    print(item)

Current project directory exists: True

Project contents:
.git
.gitattributes
.gitignore
.vscode
README.md
notebooks
scripts
src
vendor.lock


In [6]:
from pathlib import Path

lock_file = Path("/content/seis-sgm/vendor.lock")

print(lock_file.read_text())

# name  url  commit  (authors' repos, pinned so every session runs identical code)
IPS-RNA     https://github.com/mengchuangji/IPS-RNA.git     eed7523e843e513c6eb3247d7caa7324e5f41191
VI-Non-IID  https://github.com/mengchuangji/VI-Non-IID.git  6a94b8d1382974a23cb07a31c2f80f085e4094f6



In [7]:
from pathlib import Path

script = Path("/content/seis-sgm/scripts/fetch_vendor.py")

print(script.read_text())

"""Fetch pinned shallow copies of the authors' repos into ./vendor (git-ignored).
Usage:  python scripts/fetch_vendor.py [IPS-RNA] [VI-Non-IID]     (no args = all)
Stdlib only, so it runs the same on Windows and Colab."""
import pathlib
import subprocess
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
VENDOR = ROOT / "vendor"


def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True,
                          capture_output=True, text=True).stdout.strip()


def main():
    want = set(sys.argv[1:])
    for line in (ROOT / "vendor.lock").read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name, url, sha = line.split()
        if want and name not in want:
            continue
        dest = VENDOR / name
        if (dest / ".git").exists() and git("rev-parse", "HEAD", cwd=dest) == sha:
            print(f"[ok] {name} already at {sha[:8]}")
            continue
      

In [8]:
from pathlib import Path
import subprocess

root = Path("/content/seis-sgm")
vendor = root / "vendor"

print("Vendor directory exists:", vendor.exists())

if vendor.exists():
    for item in sorted(vendor.iterdir()):
        print(f"\n{item.name}/")
        print("  Is Git repository:", (item / ".git").exists())

        if (item / ".git").exists():
            result = subprocess.run(
                ["git", "rev-parse", "HEAD"],
                cwd=item,
                capture_output=True,
                text=True
            )
            print("  Current commit:", result.stdout.strip() or result.stderr.strip())
        else:
            print("  Existing contents:", [x.name for x in item.iterdir()][:20])
else:
    print("No existing vendor directory. Safe to inspect for a fresh download.")

Vendor directory exists: False
No existing vendor directory. Safe to inspect for a fresh download.


In [13]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/seis_sgm")

print("Drive directory exists:", drive_root.exists())

if drive_root.exists():
    print("\nContents:")
    for item in sorted(drive_root.iterdir()):
        print(item.name)

Drive directory exists: False
